<a href="https://colab.research.google.com/github/noelmtv/Colab-Learning/blob/main/Fundamental_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# =============================================================================
# FUNDAMENTAL ANALYSIS DASHBOARD — Google Colab Notebook
# =============================================================================
# Data source: Financial Modeling Prep (FMP) API — /stable/ endpoints
# Author: Noel Mutevhe
# =============================================================================

# Setup and API Key

In [24]:
!pip install plotly pandas requests --quiet

import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

# Ask for API key securely
from google.colab import userdata
FMP_API_KEY = userdata.get('FMP')
BASE_URL = "https://financialmodelingprep.com/stable"

def fmp_get(endpoint: str, params: dict = None) -> list | dict:
    """
    Generic FMP API caller with error handling for premium restrictions.
    Returns parsed JSON or an empty list/dict on failure.
    """
    url = f"{BASE_URL}/{endpoint}"
    all_params = {"apikey": FMP_API_KEY}
    if params:
        all_params.update(params)
    try:
        resp = requests.get(url, params=all_params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        # FMP sometimes returns a dict with "Error Message"
        if isinstance(data, dict) and "Error Message" in data:
            print(f"⚠️  {endpoint}: {data['Error Message']}")
            return [] if 'statement' in endpoint or 'segmentation' in endpoint or 'earnings' in endpoint or 'ratios' in endpoint or 'metrics' in endpoint else {}
        return data
    except requests.exceptions.HTTPError as e:
        if resp.status_code == 402:
            print(f"🔒 {endpoint}: Requires a Premium FMP subscription. Skipping.")
        else:
            print(f"❌ {endpoint}: HTTP {resp.status_code} — {e}")
        return [] if 'statement' in endpoint or 'segmentation' in endpoint or 'earnings' in endpoint or 'ratios' in endpoint or 'metrics' in endpoint else {}
    except Exception as e:
        print(f"❌ {endpoint}: {e}")
        return []

def safe_df(data, columns=None):
    """Convert API response to DataFrame, optionally selecting columns."""
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    if columns:
        existing = [c for c in columns if c in df.columns]
        df = df[existing]
    return df

def fmt_large_number(x):
    """Format large numbers for tables."""
    if pd.isna(x):
        return "—"
    if isinstance(x, (int, float)):
        if abs(x) >= 1e12:
            return f"${x/1e12:.2f}T"
        if abs(x) >= 1e9:
            return f"${x/1e9:.2f}B"
        if abs(x) >= 1e6:
            return f"${x/1e6:.2f}M"
        if abs(x) >= 1e3:
            return f"${x/1e3:.2f}K"
        return f"${x:,.2f}"
    return str(x)

print("✅ Setup complete. API key loaded securely.")

# Input Ticker
TICKER = input("Enter stock ticker (e.g., AAPL): ").strip().upper()
if not TICKER:
    TICKER = "AAPL"
    print(f"No ticker entered. Defaulting to {TICKER}.")


✅ Setup complete. API key loaded securely.
Enter stock ticker (e.g., AAPL): TSLA


# Company Profile

In [25]:
display(Markdown(f"## 📋 Company Profile — {TICKER}"))

profile_data = fmp_get("profile", {"symbol": TICKER})
if profile_data and isinstance(profile_data, list):
    p = profile_data[0]
    profile_df = pd.DataFrame([{
        "Field": k,
        "Value": v
    } for k, v in p.items() if k not in ["description", "image"]])
    # Style the profile table
    display(profile_df.style.set_properties(**{'text-align': 'left'})
            .hide(axis='index')
            .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}]))
    if p.get("description"):
        display(Markdown(f"**Business Description:** {p['description']}"))
else:
    print("⚠️  Could not retrieve company profile.")

## 📋 Company Profile — TSLA

Field,Value
symbol,TSLA
price,374.580000
marketCap,1479421462845
beta,1.916000
lastDividend,0
range,297.38-498.83
change,-11.890000
changePercentage,-3.076560
volume,51511993
averageVolume,39015254


**Business Description:** Tesla, Inc. operates globally, specializing in the creation, production, and distribution of electric vehicles, alongside comprehensive energy generation and storage solutions. Its market reach extends across the United States, China, and various other international regions. The company's operations are primarily divided into two main segments: its Automotive business and its Energy Generation and Storage division. Within its Automotive division, Tesla not only provides a range of electric cars but also generates revenue from selling automotive regulatory credits. This segment further encompasses a variety of post-sale services, including non-warranty vehicle support, sales of pre-owned vehicles, various retail products, and car insurance offerings. Customers can acquire Tesla's sedans and sport utility vehicles through direct sales, purchases of used vehicles, or via in-app upgrades often facilitated by the extensive Tesla Supercharger network. The company supports these acquisitions with financing and leasing options. Furthermore, it ensures vehicle upkeep through its proprietary service centers and a fleet of mobile technicians, complemented by both standard and extended vehicle warranty programs. The Energy Generation and Storage segment focuses on the development, manufacturing, setup, sale, and rental of solar power systems and energy storage products, along with associated services. This caters to a diverse clientele, spanning residential users, commercial enterprises, industrial entities, and public utilities. Distribution channels include Tesla's online platform, physical stores, galleries, and a network of collaborative partners. The company also offers servicing and repairs for its energy products, including warranty support, and provides multiple financing avenues for those investing in its solar solutions. Founded in 2003, the corporation was initially named Tesla Motors, Inc., before officially rebranding to Tesla, Inc. in February 2017. Its corporate headquarters are situated in Austin, Texas.

# Quick Summary

In [26]:
display(Markdown("## 🧾 Quick Summary"))

if profile_data and isinstance(profile_data, list):
    p = profile_data[0]
    summary = pd.DataFrame({
        "Metric": ["Company", "Sector", "Industry", "Market Cap", "Price", "Beta"],
        "Value": [
            p.get("companyName", "—"),
            p.get("sector", "—"),
            p.get("industry", "—"),
            fmt_large_number(p.get("marketCap")),
            f"${p.get('price', '—')}" if p.get("price") else "—",
            p.get("beta", "—")
        ]
    })
    display(summary.style.hide(axis='index'))

print("\n✅ Fundamental analysis dashboard complete.")
print("💡 Tip: If some sections are empty, they may require a Premium FMP subscription.")

## 🧾 Quick Summary

Metric,Value
Company,"Tesla, Inc."
Sector,Consumer Cyclical
Industry,Auto - Manufacturers
Market Cap,$1.48T
Price,$374.58
Beta,1.916000



✅ Fundamental analysis dashboard complete.
💡 Tip: If some sections are empty, they may require a Premium FMP subscription.


# Earnings History

In [27]:
display(Markdown(f"## 📈 Earnings History — {TICKER}"))

earnings_raw = fmp_get("earnings", {"symbol": TICKER, "limit": 40})
earnings_df = safe_df(earnings_raw)

if not earnings_df.empty:
    # Clean and sort
    earnings_df["date"] = pd.to_datetime(earnings_df["date"], errors="coerce")
    earnings_df = earnings_df.sort_values("date").reset_index(drop=True)

    # Determine available EPS columns
    eps_cols = [c for c in earnings_df.columns if "eps" in c.lower()]
    rev_cols = [c for c in earnings_df.columns if "revenue" in c.lower()]

    # Interactive earnings surprise chart (if both actual and estimated exist)
    actual_col = next((c for c in eps_cols if "actual" in c.lower()), None)
    est_col = next((c for c in eps_cols if "estimated" in c.lower()), None)

    if actual_col and est_col:
        fig_eps = go.Figure()
        fig_eps.add_trace(go.Bar(
            x=earnings_df["date"], y=earnings_df[actual_col],
            name="EPS Actual", marker_color="#2E86AB"
        ))
        fig_eps.add_trace(go.Scatter(
            x=earnings_df["date"], y=earnings_df[est_col],
            mode="lines+markers", name="EPS Estimate",
            line=dict(color="#F18F01", dash="dash")
        ))
        fig_eps.update_layout(
            title=f"{TICKER} — EPS Actual vs. Estimate",
            xaxis_title="Date", yaxis_title="EPS ($)",
            hovermode="x unified", template="plotly_white",
            legend=dict(orientation="h", yanchor="bottom", y=1.02)
        )
        fig_eps.show()
    else:
        # Fallback: plot whatever EPS column is available
        plot_col = eps_cols[0] if eps_cols else None
        if plot_col:
            fig = px.bar(earnings_df, x="date", y=plot_col,
                         title=f"{TICKER} — {plot_col}",
                         template="plotly_white")
            fig.show()

    # Display clean earnings table
    show_cols = ["date"] + [c for c in eps_cols + rev_cols if c in earnings_df.columns]
    display(earnings_df[show_cols].tail(12).style.format(precision=2))
else:
    print("⚠️  No earnings data available (may require Premium).")

## 📈 Earnings History — TSLA

🔒 earnings: Requires a Premium FMP subscription. Skipping.
⚠️  No earnings data available (may require Premium).


#Revenue Breakdown

In [28]:
# Inspect the original FMP response before normalization.

print("PRODUCT SEGMENTATION — RAW RESPONSE")
print(type(product_raw))

if isinstance(product_raw, list):
    print("Number of records:", len(product_raw))

    if product_raw:
        print("First record:")
        print(product_raw[0])

elif isinstance(product_raw, dict):
    print("Response keys:", product_raw.keys())
    print(product_raw)

else:
    print(product_raw)

PRODUCT SEGMENTATION — RAW RESPONSE
<class 'list'>
Number of records: 16
First record:
{'symbol': 'AAPL', 'fiscalYear': 2025, 'period': 'FY', 'reportedCurrency': 'USD', 'date': '2025-09-27', 'data': {'Mac': 33708000000, 'Service': 109158000000, 'Wearables, Home and Accessories': 35686000000, 'iPad': 28023000000, 'iPhone': 209586000000}}


In [29]:
# =============================================================================
# SECTION 5: PRODUCT AND GEOGRAPHIC REVENUE SEGMENTATION
# =============================================================================

import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown


display(Markdown(f"# Revenue Segmentation — {TICKER}"))


# =============================================================================
# 5.1 NORMALIZE FMP REVENUE SEGMENTATION DATA
# =============================================================================

def normalize_fmp_segmentation(raw_data):
    """
    Convert FMP revenue segmentation data into a clean DataFrame.

    FMP's stable segmentation endpoints can return records in this format:

    [
        {
            "symbol": "AAPL",
            "fiscalYear": 2025,
            "period": "FY",
            "reportedCurrency": "USD",
            "date": "2025-09-27",
            "data": {
                "iPhone": 209586000000,
                "Mac": 33708000000,
                "Services": 109158000000
            }
        }
    ]

    This function extracts the nested revenue values and combines them
    with the corresponding reporting dates and metadata.

    Parameters
    ----------
    raw_data : list, dict, or DataFrame
        Raw response from fmp_get().

    Returns
    -------
    pandas.DataFrame
        One row per reporting period, with one column per segment.
    """

    if raw_data is None:
        return pd.DataFrame()

    # Convert a DataFrame back into records if necessary.
    if isinstance(raw_data, pd.DataFrame):
        records = raw_data.to_dict(orient="records")

    elif isinstance(raw_data, dict):

        # Handle APIs or wrappers that return the records inside a key.
        if isinstance(raw_data.get("data"), list):
            records = raw_data["data"]

        elif isinstance(raw_data.get("results"), list):
            records = raw_data["results"]

        else:
            records = [raw_data]

    elif isinstance(raw_data, list):
        records = raw_data

    else:
        print(f"Unexpected response type: {type(raw_data)}")
        return pd.DataFrame()

    if not records:
        return pd.DataFrame()

    normalized_records = []

    for record in records:

        # Skip unexpected record types.
        if not isinstance(record, dict):
            continue

        # Extract metadata separately from revenue segments.
        normalized_record = {
            "date": record.get("date"),
            "fiscalYear": record.get("fiscalYear"),
            "period": record.get("period"),
            "symbol": record.get("symbol"),
            "reportedCurrency": record.get("reportedCurrency")
        }

        # FMP normally stores individual revenue segments in "data".
        nested_data = record.get("data")

        if isinstance(nested_data, dict):

            # Add the individual products or geographic regions.
            for segment_name, revenue in nested_data.items():
                normalized_record[segment_name] = revenue

        else:
            # Fallback for responses already in flat format.
            metadata_columns = {
                "date",
                "fiscalYear",
                "period",
                "symbol",
                "reportedCurrency",
                "data"
            }

            for column, value in record.items():
                if column not in metadata_columns:
                    normalized_record[column] = value

        normalized_records.append(normalized_record)

    df = pd.DataFrame(normalized_records)

    if df.empty:
        return df

    # Parse reporting dates.
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Sort by date, with older reporting periods first.
    if "date" in df.columns:
        df = df.sort_values("date", na_position="last")

    # Remove duplicate reporting dates, if any.
    # Keep the final occurrence rather than counting a period twice.
    if "date" in df.columns and df["date"].notna().any():
        df = df.drop_duplicates(subset=["date"], keep="last")

    return df.reset_index(drop=True)


# =============================================================================
# 5.2 PREPARE THE DATA FOR CHARTING
# =============================================================================

def prepare_segmentation_for_chart(df):
    """
    Identify numeric revenue columns and convert the DataFrame to long format.

    Metadata such as symbol, fiscalYear, period and reportedCurrency
    must never be treated as revenue segments.
    """

    if df.empty:
        return pd.DataFrame(), []

    # Explicit list of columns that describe the reporting record,
    # rather than individual revenue segments.
    metadata_columns = {
        "date",
        "fiscalYear",
        "period",
        "symbol",
        "reportedCurrency",
        "currency",
        "calendarYear",
        "acceptedDate",
        "fillingDate",
        "link",
        "finalLink",
        "data"
    }

    # Select candidate segment columns.
    candidate_columns = [
        column
        for column in df.columns
        if column not in metadata_columns
    ]

    # Convert candidate columns to numeric values.
    # This also handles revenue values that arrive as strings.
    for column in candidate_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # Keep columns containing at least one valid numeric observation.
    segment_columns = [
        column
        for column in candidate_columns
        if df[column].notna().any()
    ]

    # Reject negative values for a conventional revenue composition chart.
    # Negative reported values can arise in unusual restatements or
    # accounting adjustments, so we flag them rather than silently plot them.
    negative_columns = [
        column
        for column in segment_columns
        if (df[column].dropna() < 0).any()
    ]

    if negative_columns:
        print(
            "Warning: Negative values found in these segments:",
            negative_columns
        )

    if not segment_columns:
        return pd.DataFrame(), []

    # Use the actual reporting date whenever available.
    if "date" in df.columns:
        date_column = "date"
    elif "fiscalYear" in df.columns:
        date_column = "fiscalYear"
    else:
        print("No reporting date or fiscal year was found.")
        return pd.DataFrame(), []

    # Melt the wide DataFrame into long format.
    # Each row will represent one segment in one reporting period.
    long_df = df.melt(
        id_vars=[date_column],
        value_vars=segment_columns,
        var_name="Segment",
        value_name="Revenue"
    )

    long_df = long_df.dropna(subset=["Revenue"])

    # Remove records with invalid reporting dates.
    if date_column == "date":
        long_df = long_df.dropna(subset=["date"])

    return long_df, segment_columns


# =============================================================================
# 5.3 CREATE REVENUE CHARTS
# =============================================================================

def plot_revenue_segmentation(
    ticker,
    raw_data,
    segmentation_type,
    currency_fallback="USD"
):
    """
    Normalize, validate and plot product or geographic revenue segmentation.

    Produces:
        1. Stacked revenue chart.
        2. Revenue composition chart (percentage).
        3. A clean underlying revenue table.
    """

    # Normalize the original FMP response.
    df = normalize_fmp_segmentation(raw_data)

    if df.empty:
        print(
            f"No segmentation records were returned for "
            f"{ticker} ({segmentation_type})."
        )
        return

    # Identify revenue columns and prepare the long-format data.
    long_df, segment_columns = prepare_segmentation_for_chart(df)

    if long_df.empty:
        print(
            f"No usable revenue segment columns were found for "
            f"{ticker} ({segmentation_type})."
        )

        print("Available columns:", df.columns.tolist())
        display(df.head())

        return

    # Identify the reporting currency where possible.
    if (
        "reportedCurrency" in df.columns
        and df["reportedCurrency"].notna().any()
    ):
        currencies = df["reportedCurrency"].dropna().unique()

        if len(currencies) == 1:
            currency = currencies[0]
        else:
            currency = "reported currency varies"

    else:
        currency = currency_fallback

    # Select a suitable x-axis.
    x_column = "date" if "date" in long_df.columns else "fiscalYear"

    # -------------------------------------------------------------------------
    # CHART A: STACKED REVENUE
    # -------------------------------------------------------------------------

    fig = px.bar(
        long_df,
        x=x_column,
        y="Revenue",
        color="Segment",
        title=f"{ticker} — Revenue by {segmentation_type}",
        template="plotly_white",
        barmode="stack",
        labels={
            x_column: "Reporting Period",
            "Revenue": f"Revenue ({currency})",
            "Segment": segmentation_type
        },
        hover_data={
            "Revenue": ":,.0f"
        }
    )

    # Improve chart readability.
    fig.update_layout(
        height=600,
        xaxis_title="Financial Reporting Period",
        yaxis_title=f"Revenue ({currency})",
        legend_title=segmentation_type,
        hovermode="x unified",
        margin=dict(l=60, r=30, t=90, b=60)
    )

    fig.update_yaxes(tickformat="~s")

    fig.show()

    # -------------------------------------------------------------------------
    # CHART B: REVENUE COMPOSITION
    # -------------------------------------------------------------------------

    # Calculate the total reported revenue across the segments for each period.
    # This is the sum of reported segments, not necessarily total company
    # revenue. Some companies report unallocated or corporate revenue.
    composition_df = long_df.copy()

    period_totals = (
        composition_df
        .groupby(x_column)["Revenue"]
        .transform("sum")
    )

    # Avoid dividing by zero.
    composition_df["Revenue Share (%)"] = np.where(
        period_totals != 0,
        composition_df["Revenue"] / period_totals * 100,
        np.nan
    )

    fig_pct = px.bar(
        composition_df,
        x=x_column,
        y="Revenue Share (%)",
        color="Segment",
        title=f"{ticker} — {segmentation_type} Revenue Composition",
        template="plotly_white",
        barmode="stack",
        labels={
            x_column: "Reporting Period",
            "Revenue Share (%)": "Share of Reported Segments (%)",
            "Segment": segmentation_type
        }
    )

    fig_pct.update_layout(
        height=550,
        yaxis_title="Share of Reported Segment Revenue (%)",
        legend_title=segmentation_type,
        hovermode="x unified"
    )

    fig_pct.update_yaxes(ticksuffix="%")

    fig_pct.show()

    # -------------------------------------------------------------------------
    # TABLE: REVENUE BY REPORTING PERIOD
    # -------------------------------------------------------------------------

    display(
        Markdown(
            f"### {ticker} — Revenue by {segmentation_type} "
            f"({currency})"
        )
    )

    # Display only the reporting dates and revenue columns.
    display_columns = []

    for column in ["date", "fiscalYear", "period"]:
        if column in df.columns:
            display_columns.append(column)

    display_columns += segment_columns

    display(
        df[display_columns]
        .style
        .format(
            {
                column: "{:,.0f}"
                for column in segment_columns
            },
            na_rep="—"
        )
        .hide(axis="index")
    )

    # -------------------------------------------------------------------------
    # DIAGNOSTICS
    # -------------------------------------------------------------------------

    print(f"Ticker: {ticker}")
    print(f"Segmentation type: {segmentation_type}")
    print(f"Reporting currency: {currency}")
    print(f"Number of reporting periods: {len(df)}")
    print(f"Number of segments: {len(segment_columns)}")
    print(f"Segments identified: {segment_columns}")


# =============================================================================
# 5.4 PRODUCT REVENUE SEGMENTATION
# =============================================================================

display(Markdown("## Product Revenue Segmentation"))

product_raw = fmp_get(
    "revenue-product-segmentation",
    {
        "symbol": TICKER,
        "period": "annual"
    }
)

plot_revenue_segmentation(
    ticker=TICKER,
    raw_data=product_raw,
    segmentation_type="Product",
    currency_fallback="USD"
)


# =============================================================================
# 5.5 GEOGRAPHIC REVENUE SEGMENTATION
# =============================================================================

display(Markdown("## Geographic Revenue Segmentation"))

geo_raw = fmp_get(
    "revenue-geographic-segmentation",
    {
        "symbol": TICKER,
        "period": "annual"
    }
)

plot_revenue_segmentation(
    ticker=TICKER,
    raw_data=geo_raw,
    segmentation_type="Geographic Region",
    currency_fallback="USD"
)

# Revenue Segmentation — TSLA

## Product Revenue Segmentation

### TSLA — Revenue by Product (USD)

date,fiscalYear,period,Automotive,Energy Generation And Storage Segment,Services And Other
2012-12-31 00:00:00,2012,FY,"354,344,000",—,—
2013-12-31 00:00:00,2013,FY,—,—,"27,300,000"
2014-12-31 00:00:00,2014,FY,"3,007,012,000","4,208,000","187,136,000"
2015-12-31 00:00:00,2015,FY,"14,477,000","3,740,973,000","290,575,000"
2016-12-31 00:00:00,2016,FY,"6,818,738,000","181,394,000",—
2017-12-31 00:00:00,2017,FY,"9,641,300,000","1,116,266,000","1,001,185,000"
2018-12-31 00:00:00,2018,FY,"18,514,983,000","155,244,000","1,391,041,000"
2019-12-31 00:00:00,2019,FY,"20,821,000,000","1,531,000,000","2,226,000,000"
2020-12-31 00:00:00,2020,FY,"27,263,000,000","1,994,000,000","2,306,000,000"
2021-12-31 00:00:00,2021,FY,"47,232,000,000","2,789,000,000","3,802,000,000"


Ticker: TSLA
Segmentation type: Product
Reporting currency: USD
Number of reporting periods: 14
Number of segments: 3
Segments identified: ['Automotive', 'Energy Generation And Storage Segment', 'Services And Other']


## Geographic Revenue Segmentation

### TSLA — Revenue by Geographic Region (USD)

date,fiscalYear,period,CHINA,Other Countries,UNITED STATES,NETHERLANDS,NORWAY,Asia,Europe,North America
2012-12-31 00:00:00,2012,FY,—,—,—,—,—,"7,613,000","50,318,000","355,325,000"
2013-12-31 00:00:00,2013,FY,—,—,—,—,—,"1,004,000","467,079,000","1,545,413,000"
2014-12-31 00:00:00,2014,FY,"477,082,000","837,433,000","1,471,643,000",—,"412,198,000",—,—,—
2015-12-31 00:00:00,2015,FY,"318,513,000","1,413,696,000","1,957,397,000",—,"356,419,000",—,—,—
2016-12-31 00:00:00,2016,FY,"1,065,255,000","1,398,599,000","4,200,706,000",—,"335,572,000",—,—,—
2017-12-31 00:00:00,2017,FY,"2,027,062,000","2,687,169,000","6,221,439,000",—,"823,081,000",—,—,—
2018-12-31 00:00:00,2018,FY,"1,757,147,000","3,054,288,000","14,871,507,000","965,596,000","812,730,000",—,—,—
2019-12-31 00:00:00,2019,FY,"2,979,000,000","6,155,000,000","12,653,000,000","1,590,000,000","1,201,000,000",—,—,—
2020-12-31 00:00:00,2020,FY,"6,662,000,000","9,667,000,000","15,207,000,000",—,—,—,—,—
2021-12-31 00:00:00,2021,FY,"13,844,000,000","16,006,000,000","23,973,000,000",—,—,—,—,—


Ticker: TSLA
Segmentation type: Geographic Region
Reporting currency: USD
Number of reporting periods: 14
Number of segments: 8
Segments identified: ['CHINA', 'Other Countries', 'UNITED STATES', 'NETHERLANDS', 'NORWAY', 'Asia', 'Europe', 'North America']


# Financial Statements

In [30]:
display(Markdown(f"## 📊 Financial Statements — {TICKER}"))

STATEMENTS = {
    "Income Statement": "income-statement",
    "Balance Sheet": "balance-sheet-statement",
    "Cash Flow": "cash-flow-statement"
}

for label, endpoint in STATEMENTS.items():
    display(Markdown(f"### {label}"))
    raw = fmp_get(endpoint, {"symbol": TICKER, "period": "annual", "limit": 5})
    df = safe_df(raw)
    if not df.empty:
        # Sort by date descending for latest first
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df = df.sort_values("date", ascending=False)
        display(df.style.format(precision=0).hide(axis='index'))
    else:
        print(f"⚠️  {label} not available (may require Premium).")

## 📊 Financial Statements — TSLA

### Income Statement

date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,generalAndAdministrativeExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,otherExpenses,operatingExpenses,costAndExpenses,netInterestIncome,interestIncome,interestExpense,depreciationAndAmortization,ebitda,ebit,nonOperatingIncomeExcludingInterest,operatingIncome,totalOtherIncomeExpensesNet,incomeBeforeTax,incomeTaxExpense,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,netIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil
2025-12-31 00:00:00,TSLA,USD,0001318605,2026-01-29,2026-01-28 20:55:03,2025,FY,94827000000,77733000000,17094000000,6411000000,5834000000,0,5834000000,494000000,12739000000,90472000000,1342000000,1680000000,338000000,6148000000,11764000000,5616000000,-1261000000,4355000000,923000000,5278000000,1423000000,3855000000,0,0,3794000000,0,3794000000,1,1,3225000000,3528000000
2024-12-31 00:00:00,TSLA,USD,0001318605,2025-01-30,2025-01-29 20:42:33,2024,FY,97690000000,80240000000,17450000000,4540000000,5150000000,0,5150000000,684000000,10374000000,90614000000,1219000000,1569000000,350000000,5368000000,14708000000,9340000000,-2264000000,7076000000,1914000000,8990000000,1837000000,7153000000,0,0,7130000000,0,7130000000,2,2,3197000000,3498000000
2023-12-31 00:00:00,TSLA,USD,0001318605,2024-01-29,2024-01-28 19:00:00,2023,FY,96773000000,79113000000,17660000000,3969000000,4800000000,0,4800000000,0,8769000000,87882000000,910000000,1066000000,156000000,4667000000,14796000000,10129000000,-1238000000,8891000000,1082000000,9973000000,-5001000000,14974000000,0,2000000,14999000000,0,14999000000,5,4,3173500000,3482750000
2022-12-31 00:00:00,TSLA,USD,0001318605,2023-01-31,2023-01-30 21:29:15,2022,FY,81462000000,60609000000,20853000000,3075000000,3946000000,0,3946000000,176000000,7197000000,67806000000,106000000,297000000,191000000,3543000000,17453000000,13910000000,-254000000,13656000000,63000000,13719000000,1132000000,12587000000,0,27000000,12583000000,-1000000,12584000000,4,4,3130000000,3475000000
2021-12-31 00:00:00,TSLA,USD,0001318605,2022-02-07,2022-02-06 19:00:00,2021,FY,53823000000,40217000000,13606000000,2593000000,4517000000,0,4517000000,0,7110000000,47327000000,-315000000,56000000,371000000,2911000000,9625000000,6714000000,-191000000,6523000000,-180000000,6343000000,699000000,5644000000,0,5000000,5524000000,-4000000,5533000000,2,2,2959000000,3386000000


### Balance Sheet

date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,cashAndCashEquivalents,shortTermInvestments,cashAndShortTermInvestments,netReceivables,accountsReceivables,otherReceivables,inventory,prepaids,otherCurrentAssets,totalCurrentAssets,propertyPlantEquipmentNet,goodwill,intangibleAssets,goodwillAndIntangibleAssets,longTermInvestments,taxAssets,otherNonCurrentAssets,totalNonCurrentAssets,otherAssets,totalAssets,totalPayables,accountPayables,otherPayables,accruedExpenses,shortTermDebt,capitalLeaseObligationsCurrent,taxPayables,deferredRevenue,otherCurrentLiabilities,totalCurrentLiabilities,longTermDebt,capitalLeaseObligationsNonCurrent,deferredRevenueNonCurrent,deferredTaxLiabilitiesNonCurrent,otherNonCurrentLiabilities,totalNonCurrentLiabilities,otherLiabilities,capitalLeaseObligations,totalLiabilities,treasuryStock,preferredStock,commonStock,retainedEarnings,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt
2025-12-31 00:00:00,TSLA,USD,0001318605,2026-01-29,2026-01-28 20:55:03,2025,FY,16513000000,27546000000,44059000000,4576000000,4576000000,0,12392000000,0,7615000000,68642000000,40643000000,257000000,135000000,392000000,0,6925000000,21204000000,69164000000,0,137806000000,13371000000,13371000000,0,3785000000,1640000000,0,0,3424000000,9494000000,31714000000,6736000000,0,3631000000,0,12860000000,23227000000,0,0,54941000000,0,0,3000000,39003000000,42770000000,361000000,0,82137000000,82865000000,728000000,137806000000,27546000000,8376000000,-8137000000
2024-12-31 00:00:00,TSLA,USD,0001318605,2025-01-30,2025-01-29 20:42:33,2024,FY,16139000000,20424000000,36563000000,4418000000,4418000000,0,12017000000,0,5362000000,58360000000,51507000000,244000000,1226000000,1470000000,0,6524000000,4215000000,63716000000,-6000000,122070000000,13841000000,12474000000,1367000000,3785000000,2343000000,920000000,1367000000,4161000000,3771000000,28821000000,5535000000,4825000000,3317000000,0,5892000000,19569000000,0,5745000000,48390000000,0,0,3000000,35209000000,38371000000,-670000000,0,72913000000,73680000000,767000000,122070000000,20424000000,13623000000,-2516000000
2023-12-31 00:00:00,TSLA,USD,0001318605,2024-01-29,2024-01-28 19:00:00,2023,FY,16398000000,12696000000,29094000000,3508000000,3508000000,0,13626000000,0,3388000000,49616000000,45123000000,253000000,362000000,615000000,0,6733000000,4532000000,57003000000,-1000000,106618000000,15635000000,14431000000,1204000000,4046000000,1975000000,1070000000,1204000000,3740000000,2282000000,28748000000,2682000000,3846000000,3251000000,0,4482000000,14261000000,0,4916000000,43009000000,0,0,3000000,27882000000,34892000000,-143000000,0,62634000000,63609000000,975000000,106618000000,12696000000,9573000000,-6825000000
2022-12-31 00:00:00,TSLA,USD,0001318605,2023-01-31,2023-01-30 21:29:15,2022,FY,16253000000,5932000000,22185000000,2952000000,2952000000,0,12839000000,0,2941000000,40917000000,36635000000,194000000,399000000,593000000,657000000,328000000,3208000000,41421000000,0,82338000000,16490000000,15255000000,1235000000,3773000000,1016000000,971000000,1235000000,2810000000,1649000000,26709000000,1029000000,2732000000,2804000000,82000000,3084000000,9731000000,0,3703000000,36440000000,0,0,3000000,12885000000,32177000000,-361000000,0,44704000000,45898000000,1194000000,82338000000,6589000000,5748000000,-10505000000
2021-12-31 00:00:00,TSLA,USD,0001318605,2022-02-07,2022-02-06 19:00:00,2021,FY,17576000000,131000000,17707000000,1913000000,1913000000,0,5757000000,1723000000,0,27100000000,31176000000,200000000,1517000000,1717000000,522000000,0,1612000000,35027000000,4000000,62131000000,11147000000,10025000000,1122000000,2951000000,1088000000,869000000,1122000000,2372000000,1278000000,19705000000,4254000000,2662000000,2052000000,24000000,1851000000,10843000000,0,3531000000,30548000000,0,0,3000000,329000000,29803000000,54000000,0,30189000000,

### Cash Flow

date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,netIncome,depreciationAndAmortization,deferredIncomeTax,stockBasedCompensation,changeInWorkingCapital,accountsReceivables,inventory,accountsPayables,otherWorkingCapital,otherNonCashItems,netCashProvidedByOperatingActivities,investmentsInPropertyPlantAndEquipment,acquisitionsNet,purchasesOfInvestments,salesMaturitiesOfInvestments,otherInvestingActivities,netCashProvidedByInvestingActivities,netDebtIssuance,longTermNetDebtIssuance,shortTermNetDebtIssuance,netStockIssuance,netCommonStockIssuance,commonStockIssuance,commonStockRepurchased,netPreferredStockIssuance,netDividendsPaid,commonDividendsPaid,preferredDividendsPaid,otherFinancingActivities,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid
2025-12-31 00:00:00,TSLA,USD,0001318605,2026-01-29,2026-01-28 20:55:03,2025,FY,3855000000,6148000000,123000000,2825000000,642000000,-261000000,-630000000,4376000000,-2843000000,1154000000,14747000000,-8527000000,0,-37109000000,30158000000,0,-15478000000,-3158000000,-3158000000,0,222000000,222000000,222000000,0,0,0,0,0,4075000000,1139000000,171000000,579000000,17616000000,17037000000,14747000000,-8527000000,6220000000,0,0
2024-12-31 00:00:00,TSLA,USD,0001318605,2025-01-30,2025-01-29 20:42:33,2024,FY,7130000000,5368000000,477000000,1999000000,81000000,-1083000000,937000000,3588000000,-3361000000,-155000000,14923000000,-11342000000,0,-35955000000,28510000000,0,-18787000000,2863000000,2863000000,0,1241000000,1241000000,1241000000,0,0,0,0,0,-251000000,3853000000,-141000000,-152000000,17037000000,17189000000,14923000000,-11342000000,3581000000,1331000000,277000000
2023-12-31 00:00:00,TSLA,USD,0001318605,2024-01-29,2024-01-28 19:00:00,2023,FY,14999000000,4667000000,-6349000000,1812000000,-2248000000,-586000000,-1195000000,2605000000,-3072000000,400000000,13256000000,-8899000000,-64000000,-19112000000,12491000000,0,-15584000000,2116000000,2116000000,0,700000000,700000000,700000000,0,0,0,0,0,-227000000,2589000000,4000000,265000000,17189000000,16924000000,13256000000,-8899000000,4357000000,1119000000,126000000
2022-12-31 00:00:00,TSLA,USD,0001318605,2023-01-31,2023-01-30 21:29:15,2022,FY,12583000000,3543000000,0,1560000000,-3712000000,-1124000000,-6465000000,6029000000,-2152000000,750000000,14724000000,-7163000000,0,-5835000000,22000000,1003000000,-11973000000,-3866000000,-3866000000,0,541000000,541000000,541000000,0,0,0,0,0,-202000000,-3527000000,-444000000,-1220000000,16924000000,18144000000,14724000000,-7172000000,7552000000,1203000000,152000000
2021-12-31 00:00:00,TSLA,USD,0001318605,2022-02-07,2022-02-06 19:00:00,2021,FY,5519000000,2911000000,-149000000,2121000000,667000000,-130000000,-1709000000,4578000000,-2072000000,428000000,11497000000,-6514000000,0,-132000000,0,-1222000000,-7868000000,-5732000000,-5732000000,0,707000000,707000000,707000000,0,0,0,0,0,-178000000,-5203000000,-183000000,-1757000000,18144000000,19901000000,11497000000,-8014000000,3483000000,561000000,266000000


# EPS History

In [31]:
display(Markdown(f"## 💹 EPS & P/E History — {TICKER}"))

# A. Get historical prices
price_raw = fmp_get("historical-price-eod/full", {
    "symbol": TICKER,
    "from": (pd.Timestamp.now() - pd.DateOffset(years=5)).strftime("%Y-%m-%d"),
    "to": pd.Timestamp.now().strftime("%Y-%m-%d")
})
price_df = safe_df(price_raw)

# B. Get earnings EPS (annual)
eps_raw = fmp_get("earnings", {"symbol": TICKER, "limit": 20})
eps_df = safe_df(eps_raw)

if not price_df.empty and not eps_df.empty:
    price_df["date"] = pd.to_datetime(price_df["date"], errors="coerce")
    eps_df["date"] = pd.to_datetime(eps_df["date"], errors="coerce")

    # Find EPS actual column
    eps_col = next((c for c in eps_df.columns if "eps" in c.lower() and "actual" in c.lower()),
                   next((c for c in eps_df.columns if "eps" in c.lower()), None))

    if eps_col:
        eps_series = eps_df.set_index("date")[eps_col].sort_index()

        # Compute trailing P/E at each price date using the latest reported EPS
        price_df = price_df.sort_values("date")
        price_df["eps_reported"] = np.nan
        for idx, row in price_df.iterrows():
            # Find most recent EPS report on or before this date
            prior_eps = eps_series[eps_series.index <= row["date"]]
            if not prior_eps.empty:
                price_df.at[idx, "eps_reported"] = prior_eps.iloc[-1]

        price_df["pe_ratio"] = price_df["close"] / price_df["eps_reported"]
        price_df = price_df.dropna(subset=["pe_ratio"])

        if not price_df.empty:
            fig_pe = make_subplots(specs=[[{"secondary_y": True}]])
            fig_pe.add_trace(
                go.Scatter(x=price_df["date"], y=price_df["pe_ratio"],
                           name="P/E Ratio", line=dict(color="#2E86AB")),
                secondary_y=False
            )
            # Overlay EPS bars on secondary axis (annual points)
            fig_pe.add_trace(
                go.Bar(x=eps_series.index, y=eps_series.values,
                       name="EPS (Reported)", marker_color="#F18F01", opacity=0.6),
                secondary_y=True
            )
            fig_pe.update_layout(
                title=f"{TICKER} — Historical P/E vs. Reported EPS",
                hovermode="x unified", template="plotly_white",
                legend=dict(orientation="h", yanchor="bottom", y=1.02)
            )
            fig_pe.update_yaxes(title_text="P/E Ratio", secondary_y=False)
            fig_pe.update_yaxes(title_text="EPS ($)", secondary_y=True)
            fig_pe.show()
        else:
            print("⚠️  Could not compute P/E series.")
    else:
        print("⚠️  No EPS column found for P/E calculation.")
else:
    print("⚠️  Price or earnings data insufficient for P/E history.")

## 💹 EPS & P/E History — TSLA

🔒 earnings: Requires a Premium FMP subscription. Skipping.
⚠️  Price or earnings data insufficient for P/E history.


# Key Metrics and Ratios

In [32]:
display(Markdown(f"## 📐 Key Metrics & Ratios (TTM) — {TICKER}"))

# a. Ratios TTM
ratios_raw = fmp_get("ratios-ttm", {"symbol": TICKER})
ratios_df = safe_df(ratios_raw)

# b. Key Metrics TTM
metrics_raw = fmp_get("key-metrics-ttm", {"symbol": TICKER})
metrics_df = safe_df(metrics_raw)

if not ratios_df.empty:
    # Transpose single-row ratios for readability
    ratios_t = ratios_df.T.reset_index()
    ratios_t.columns = ["Metric", "Value"]
    display(ratios_t.style.format(precision=3).hide(axis='index'))
else:
    print("⚠️  Ratios TTM not available (may require Premium).")

if not metrics_df.empty:
    metrics_t = metrics_df.T.reset_index()
    metrics_t.columns = ["Metric", "Value"]
    display(metrics_t.style.format(precision=3).hide(axis='index'))
else:
    print("⚠️  Key Metrics TTM not available (may require Premium).")

## 📐 Key Metrics & Ratios (TTM) — TSLA

Metric,Value
symbol,TSLA
grossProfitMarginTTM,0.193
ebitMarginTTM,0.044
ebitdaMarginTTM,0.109
operatingProfitMarginTTM,0.033
pretaxProfitMarginTTM,0.041
continuousOperationsProfitMarginTTM,0.032
netProfitMarginTTM,0.031
bottomLineProfitMarginTTM,0.031
receivablesTurnoverTTM,22.340


Metric,Value
symbol,TSLA
marketCap,1479421462845
enterpriseValueTTM,1478013462845
evToSalesTTM,14.271
evToOperatingCashFlowTTM,78.996
evToFreeCashFlowTTM,447.476
evToEBITDATTM,131.192
netDebtToEBITDATTM,-0.125
currentRatioTTM,1.739
incomeQualityTTM,5.734
